# 2.1 Quality Fixes

Comprueba la etapa `quality_fixes` sobre la partición `train`
(2024-10-01 a 2026-02-22), a partir de la etapa `structure`.

In [1]:
from mibici.pipeline import load_stage
from mibici.data_preparation.quality_fixes import (
    remove_duplicate_trips,
    replace_invalid_birth_years,
)
import pandas as pd
data = load_stage("structure")
structure_data = data

2026-10-18 09:07:46.836 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: X:\Repositorios\Personal\Kaggle\notebooks\mibici\mibici_predictions


2026-10-18 09:07:56.463 | INFO     | mibici.dataset:load_raw:166 - Partición train: 2024-10-01 a 2026-02-22
2026-10-18 09:07:56.473 | INFO     | mibici.dataset:load_raw:170 - Viajes cargados: 6,311,111


### 1.- Missing Values

In [2]:
trips = data["viajes"].copy()
stations = data["estaciones"].copy()

display(trips.isna().sum())
display(stations.isna().sum())

Viaje_Id                 0
Usuario_Id               0
Genero               53941
Año_de_nacimiento     1717
Inicio_del_viaje         0
Fin_del_viaje            0
Origen_Id                0
Destino_Id               0
dtype: int64

id            0
name          0
latitude      0
longitude     0
dpcapacity    0
dtype: int64

Faltantes en train: 53,941 en `Genero` y 1,717 en `Año_de_nacimiento`;
las demás columnas y el catálogo no tienen nulos. No se imputan.

### 2.- Duplicates

In [3]:
before = data["viajes"].copy()
after = remove_duplicate_trips(before)

assert len(after) == len(before.drop_duplicates())
assert after.duplicated().sum() == 0
assert after.columns.equals(before.columns)

print(f"Copias eliminadas: {len(before) - len(after):,}")

Copias eliminadas: 0


Train no tiene copias exactas: `remove_duplicate_trips` conserva
las 6,311,111 filas y las columnas (H-co-005).

### 3.- Domain Constraints

In [4]:
from mibici.config import MIN_AGE, MAX_AGE

before = remove_duplicate_trips(data["viajes"].copy())

after = replace_invalid_birth_years(
    before,
    min_age=MIN_AGE,
    max_age=MAX_AGE,
)

age = (
    after["Inicio_del_viaje"].dt.year
    - after["Año_de_nacimiento"]
)

assert len(after) == len(before)
assert str(after["Año_de_nacimiento"].dtype) == "Int64"
assert age.dropna().between(MIN_AGE, MAX_AGE).all()

other_columns = before.columns.drop("Año_de_nacimiento")
assert after[other_columns].equals(before[other_columns])

print(f"Años faltantes: {after['Año_de_nacimiento'].isna().sum():,}")

Años faltantes: 2,572


In [5]:
# Valores conservados: solo cambian los años fuera de [MIN_AGE, MAX_AGE]
birth_before = before["Año_de_nacimiento"]
birth_after = after["Año_de_nacimiento"]
approx_age_before = before["Inicio_del_viaje"].dt.year - birth_before

became_missing = birth_before.notna() & birth_after.isna()
expected = (
    birth_before.notna()
    & (birth_before.le(0) | approx_age_before.lt(MIN_AGE) | approx_age_before.gt(MAX_AGE))
).fillna(False).astype(bool)

assert became_missing.equals(expected)
assert birth_after[~became_missing].equals(birth_before[~became_missing])

print(f"Años convertidos a faltantes: {became_missing.sum():,}")
print(f"Faltantes de año: {birth_before.isna().sum():,} -> {birth_after.isna().sum():,}")

# Desglose por año convertido
display(
    before.loc[became_missing]
    .assign(approx_age=approx_age_before[became_missing])
    .groupby("Año_de_nacimiento")
    .agg(
        trip_count=("Viaje_Id", "size"),
        user_count=("Usuario_Id", "nunique"),
        min_age=("approx_age", "min"),
        max_age=("approx_age", "max"),
    )
)

Años convertidos a faltantes: 855
Faltantes de año: 1,717 -> 2,572


,trip_count,user_count,min_age,max_age
Año_de_nacimiento,,,,
199,164,1,1826,1827
200,5,1,1824,1824
1920,160,1,104,105
2023,526,10,1,2


Con `MIN_AGE=10` y `MAX_AGE=100` se convierten 855 años a faltantes:
2023 (526 viajes, 1–2 años), 199 (164), 1920 (160, 104–105 años) y 200 (5).
Solo cambian esos valores: el resto de los años y las demás columnas se conservan,
sin eliminar filas. Los faltantes de año pasan de 1,717 a 2,572.

Los límites siguen en revisión (H-co-006).

### 4.- Continuous Variables

In [6]:
display(stations.loc[stations["dpcapacity"].le(0)])

,id,name,latitude,longitude,dpcapacity
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,20.685108,-103.333746,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,20.723221,-103.385583,0
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,20.686920,-103.334670,0
129,138,ex-(ZPN-044),20.697050,-103.363220,0


Las estaciones 99, 109, 116 y 138 tienen capacidad cero.
La etapa no las modifica; su tratamiento está pendiente (H-co-007).

### 5.- Inconsistency

In [7]:
assert trips["Origen_Id"].isin(stations["id"]).all()
assert trips["Destino_Id"].isin(stations["id"]).all()

Todos los `Origen_Id` y `Destino_Id` de train existen en el catálogo (H-co-011).

### 6.- Temporal Relationships

In [8]:
duration = trips["Fin_del_viaje"] - trips["Inicio_del_viaje"]

assert duration.notna().all()
assert duration.ge(pd.Timedelta(0)).all()

print(f"Duraciones cero: {duration.eq(pd.Timedelta(0)).sum():,}")

Duraciones cero: 12,636


No hay duraciones negativas ni faltantes. Se conservan los 12,636 viajes
con duración cero; su tratamiento está pendiente (H-co-008).

### 7.- Constant / Near-Constant Variables

In [9]:
data = load_stage("quality_fixes")

trips = data["viajes"]
stations = data["estaciones"]

def summarize_constants(df):
    return pd.DataFrame({
        "unique_count": df.nunique(dropna=False),
        "top_value_pct": [
            df[column]
            .value_counts(normalize=True, dropna=False)
            .max() * 100
            for column in df.columns
        ],
    }).round(2)


display(summarize_constants(trips))
display(summarize_constants(stations))

2026-10-18 09:08:27.592 | INFO     | mibici.dataset:load_raw:166 - Partición train: 2024-10-01 a 2026-02-22
2026-10-18 09:08:27.592 | INFO     | mibici.dataset:load_raw:170 - Viajes cargados: 6,311,111


,unique_count,top_value_pct
Viaje_Id,6311111,0.00
Usuario_Id,36317,0.09
Genero,3,71.76
Año_de_nacimiento,71,5.17
Inicio_del_viaje,5269632,0.00
Fin_del_viaje,5283792,0.00
Origen_Id,371,1.92
Destino_Id,371,2.49


,unique_count,top_value_pct
id,484,0.21
name,484,0.21
latitude,480,0.41
longitude,481,0.41
dpcapacity,9,41.53


Después de `quality_fixes` no hay columnas constantes ni una concentración
extrema en un único valor (máximo: `Genero`, 71.76 %).

### 8.- Correspondencia con la etapa

In [10]:
# Correspondencia: funciones individuales = load_stage("quality_fixes")
pd.testing.assert_frame_equal(after, trips)
pd.testing.assert_frame_equal(structure_data["estaciones"], stations)
print("Funciones individuales = load_stage('quality_fixes')")
print(f"Viajes: {len(structure_data['viajes']):,} -> {len(trips):,}")

Funciones individuales = load_stage('quality_fixes')
Viajes: 6,311,111 -> 6,311,111


Aplicar `remove_duplicate_trips` y `replace_invalid_birth_years`
produce el mismo resultado que `load_stage("quality_fixes")`.
El catálogo de estaciones no cambia y se conservan las 6,311,111 filas.